# Phase 9: Formal Time-Series Stationarity Testing
## Dual-Hypothesis Framework (ADF + KPSS), Differencing & Feature Justification

**Objective**:
In quantitative finance and machine learning for algorithmic trading, stationarity is a foundational prerequisite for statistical inference and predictive modeling. This notebook provides the rigorous mathematical justification and empirical testing required before feature engineering (Phases 10–13).

### Core Questions Addressed:
1. **Why is Stationarity Mandatory?** Avoiding spurious regression, unbounded error variances, and breakdown of statistical laws.
2. **The Dual-Hypothesis Protocol**: Why relying on a single test (e.g. only ADF or only KPSS) is hazardous, and how the 4-quadrant decision matrix establishes consensus.
3. **Empirical Verification (AAPL, MSFT, SPY)**: Testing raw prices, log prices, arithmetic returns, and log returns.
4. **Order of Integration & Differencing**: Formally demonstrating $I(1) \to I(0)$ transformation via first-order differencing.
5. **Architectural Guidelines**: Concrete rules governing which series and transformed features will feed subsequent machine learning models.

In [ ]:
import sys
import types
from pathlib import Path

# Ensure project root is accessible
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Safeguard for environments where Application Control restricts C-extensions
if "matplotlib._c_internal_utils" not in sys.modules:
    try:
        import matplotlib._c_internal_utils  # noqa: F401
    except ImportError:
        sys.modules["matplotlib._c_internal_utils"] = types.ModuleType(
            "matplotlib._c_internal_utils"
        )

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_pipeline.data_access import get_data_access
from src.features.stationarity import (
    adf_test,
    kpss_test,
    stationarity_report,
    test_multiple_series,
    difference_series,
)

reports_dir = project_root / "reports" / "stationarity"
reports_dir.mkdir(parents=True, exist_ok=True)
print("Phase 9 Stationarity Environment Initialized.")
print("Reports directory:", reports_dir)

## 1. Theoretical Foundations: Stationarity & The Spurious Regression Hazard

### 1.1 Weak (Covariance) Stationarity
A stochastic process $\\{y_t\\}$ is weakly stationary if:
1. **Constant Mean**: $\mathbb{E}[y_t] = \mu < \infty$ for all $t$.
2. **Finite, Constant Variance**: $\text{Var}(y_t) = \sigma^2 < \infty$ for all $t$.
3. **Time-Invariant Autocovariance**: $\text{Cov}(y_t, y_{t-k}) = \gamma_k$ depends only on lag $k$, not on calendar time $t$.

### 1.2 Spurious Regression (Granger & Newbold, 1974)
When two independent non-stationary unit-root processes $y_t$ and $x_t$ are regressed against each other:
$$y_t = \alpha + \beta x_t + \epsilon_t$$
Standard OLS yields high $R^2$ values and statistically significant $t$-statistics even when $x_t$ and $y_t$ are completely unrelated. Residuals $\epsilon_t$ inherit non-stationarity, causing standard error estimates to collapse toward zero and generating false-positive trading signals.

### 1.3 The Dual-Hypothesis Framework
Single-test stationarity analysis is prone to severe inferential errors:
- **Augmented Dickey-Fuller (ADF)**:
  - $H_0$: Series possesses a unit root (non-stationary).
  - $H_1$: Series is stationary.
  - Low test power against near-unit roots (e.g. $\rho = 0.98$).
- **Kwiatkowski-Phillips-Schmidt-Shin (KPSS)**:
  - $H_0$: Series is level or trend stationary.
  - $H_1$: Series possesses a unit root (non-stationary).
  - Sensitive to structural breaks and long-memory noise.

| ADF Result ($p < 0.05$) | KPSS Result ($p \ge 0.05$) | Consensus Verdict | Theoretical Interpretation |
| :--- | :--- | :--- | :--- |
| **Reject $H_0$** (Stat) | **Fail to Reject $H_0$** (Stat) | **Stationary** | Both tests concur; series is $I(0)$ mean-reverting. |
| **Fail to Reject** (Non-Stat) | **Reject $H_0$** (Non-Stat) | **Non-Stationary** | Both tests concur; series possesses a unit root ($I(1)$). |
| **Fail to Reject** (Non-Stat) | **Fail to Reject** (Stat) | **Inconclusive** | Low test power or near-unit-root dynamics. |
| **Reject $H_0$** (Stat) | **Reject $H_0$** (Non-Stat) | **Inconclusive** | Structural breaks, regime change, or trend stationarity. |

## 2. Ingestion via Unified DataAccessLayer
Data is retrieved strictly through `DataAccessLayer` from partitioned storage (`/data/processed/{ticker}/{year}.parquet`).

In [ ]:
dal = get_data_access()
tickers = ["AAPL", "MSFT", "SPY"]
ohlcv_dict = {t: dal.get_ohlcv(t) for t in tickers}

for t, df in ohlcv_dict.items():
    start_dt = df.index[0].date()
    end_dt = df.index[-1].date()
    p_start = df["close"].iloc[0]
    p_end = df["close"].iloc[-1]
    print(f"{t:<5}: {len(df):>4} bars | {start_dt} -> {end_dt} | Closes: ${p_start:.2f} -> ${p_end:.2f}")

## 3. Visual Intuition: Non-Stationary Prices vs. Stationary Returns
We plot the raw closing prices alongside daily arithmetic returns for AAPL, MSFT, and SPY to illustrate the dramatic visual difference between non-stationary paths (wandering, stochastic drift) and stationary oscillations (mean-reverting around zero).

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex="col")

for idx, t in enumerate(tickers):
    df = ohlcv_dict[t]
    ret = df["close"].pct_change().dropna()
    
    # Price plot
    axes[idx, 0].plot(df.index, df["close"], color="#1f77b4", lw=1.2, label=f"{t} Close ($)")
    axes[idx, 0].set_title(f"{t} Raw Price (Wandering Stochastic Trend)", fontsize=11, fontweight="bold")
    axes[idx, 0].set_ylabel("Price ($)")
    axes[idx, 0].grid(True, alpha=0.3)
    axes[idx, 0].legend(loc="upper left")
    
    # Returns plot
    axes[idx, 1].plot(ret.index, ret, color="#2ca02c", lw=0.7, alpha=0.8, label=f"{t} Daily Return")
    axes[idx, 1].axhline(0, color="black", linestyle="--", lw=0.8, alpha=0.7)
    axes[idx, 1].set_title(f"{t} Daily Returns (Stationary Mean-Reverting Oscillation)", fontsize=11, fontweight="bold")
    axes[idx, 1].set_ylabel("Daily Return")
    axes[idx, 1].grid(True, alpha=0.3)
    axes[idx, 1].legend(loc="upper left")

plt.tight_layout()
fig_path = reports_dir / "price_vs_returns.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
print(f"Visual intuition figure saved to: {fig_path}")
plt.show()

## 4. Empirical Testing: Raw Prices & Log Prices ($P_t$ and $\ln P_t$)
We subject raw closing prices and natural log prices to both ADF and KPSS testing.

In [ ]:
price_series_dict = {}
for t in tickers:
    p = ohlcv_dict[t]["close"]
    price_series_dict[f"{t}_raw_price"] = p
    price_series_dict[f"{t}_log_price"] = np.log(p)

price_test_results = test_multiple_series(price_series_dict, alpha=0.05)
price_test_results

### Interpretation of Price Results:
- **ADF Statistics**: All test statistics are greater than the critical values (e.g. 1%: -3.43, 5%: -2.86). p-values are $\ge 0.53$, failing to reject the unit-root null.
- **KPSS Statistics**: Test statistics exceed the 5% critical value (0.463), resulting in p-values of $0.01$, decisively rejecting stationarity.
- **Consensus Verdict**: **`non-stationary`** across all assets. Log transformations do not remove stochastic trends; $\ln P_t$ remains non-stationary.

## 5. Empirical Testing: Arithmetic Returns & Log Returns
Next, we evaluate daily percentage returns $R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$ and log returns $r_t = \ln(P_t / P_{t-1})$.

In [ ]:
return_series_dict = {}
for t in tickers:
    p = ohlcv_dict[t]["close"]
    return_series_dict[f"{t}_arithmetic_ret"] = p.pct_change().dropna()
    return_series_dict[f"{t}_log_ret"] = np.log(p / p.shift(1)).dropna()

return_test_results = test_multiple_series(return_series_dict, alpha=0.05)
return_test_results

### Interpretation of Return Results:
- **ADF Statistics**: Test statistics are deeply negative ($\\le -14.8$), with p-values on the order of $10^{-27}$ to $10^{-28}$, decisively rejecting the unit root null.
- **KPSS Statistics**: Test statistics are small ($< 0.3$), with p-values $\ge 0.10$, failing to reject the stationarity null.
- **Consensus Verdict**: **`stationary`** across all assets. Both arithmetic and log returns are weakly stationary.

## 6. Differencing Verification & Order of Integration
A series is said to be integrated of order $d$, denoted $I(d)$, if differencing it $d$ times yields a stationary $I(0)$ process:
$$\Delta^d y_t \sim I(0)$$
Here we formally verify that first-order differencing ($d=1$) of raw prices converts them into stationary series.

In [ ]:
diff_series_dict = {}
for t in tickers:
    p = ohlcv_dict[t]["close"]
    diff_series_dict[f"{t}_diff1_price"] = difference_series(p, order=1)

diff_test_results = test_multiple_series(diff_series_dict, alpha=0.05)
diff_test_results

## 7. Comprehensive Master Summary Table
We compile all tested series into a unified stationarity benchmark table and export it for downstream documentation.

In [ ]:
master_series_dict = {}
for t in tickers:
    p = ohlcv_dict[t]["close"]
    master_series_dict[f"{t}_raw_price"] = p
    master_series_dict[f"{t}_log_price"] = np.log(p)
    master_series_dict[f"{t}_daily_ret"] = p.pct_change().dropna()
    master_series_dict[f"{t}_log_ret"] = np.log(p / p.shift(1)).dropna()
    master_series_dict[f"{t}_diff1_price"] = difference_series(p, order=1)

master_df = test_multiple_series(master_series_dict, alpha=0.05)
csv_path = reports_dir / "stationarity_summary.csv"
master_df.to_csv(csv_path)
print(f"Master Stationarity Table exported to: {csv_path}")
master_df

## 8. Summary of Findings & Architectural Directives for Feature Engineering (Phases 10–13)

### Empirical Findings:
1. **Raw Prices are $I(1)$**: Raw asset prices exhibit significant unit-root behavior ($p_{ADF} > 0.50$, $p_{KPSS} = 0.01$). Passing raw prices directly into ML models will cause spurious correlation and regime overfitting.
2. **Log Transformation Does Not Induce Stationarity**: Taking logarithms compresses scale and stabilizes variance across exponential growth, but $\ln(P_t)$ remains $I(1)$.
3. **Returns are $I(0)$**: Both arithmetic daily returns and continuous log returns are strongly stationary ($p_{ADF} < 10^{-27}$, $p_{KPSS} \ge 0.10$).
4. **First Differencing Achieves Stationarity**: First-order differencing $\Delta P_t$ eliminates the unit root, confirming that asset price processes are integrated of order 1 ($d=1$).

### Strict Architectural Directives for Downstream Pipelines:
- **Rule 1: No Unscaled Raw Prices in Features**: Features based on raw prices (e.g., Close, Moving Averages) must be converted into scale-free stationary forms, such as percentage distance from moving average: $\frac{P_t - \text{SMA}_k(P_t)}{\text{SMA}_k(P_t)}$.
- **Rule 2: Normalized Oscillators**: Indicators like RSI, Stochastic %K, and Normalized MACD oscillate within stationary bounded ranges $[0, 100]$ or around 0 and are admissible.
- **Rule 3: Volatility Normalization**: When using price differences or returns, scale by rolling ATR or standard deviation (e.g. z-scoring) to ensure stationarity in variance across volatility regimes.
- **Rule 4: Preservation of Long-Memory**: Standard integer differencing ($d=1$) removes all price-level memory. In advanced phases, fractional differencing ($0 < d < 1$) will be explored to achieve stationarity while retaining maximum predictive memory.